# ⚔️ B2　Boss 戰：心臟病風險預警
**統計冒險之旅 2026**　｜　Day 2（09/24 四）⛰️ 模型之嶺　｜　Boss 戰　｜　🏅 200 XP

📖 Day 2 綜合；資料：勇者健檢中心（固定種子合成資料）

### 🎯 這一關你會學到
- 缺值盤點 → train/dev/final test → Pipeline 補值與標準化 → 邏輯斯迴歸
- 用 dev 選門檻，final test 只報告一次
- 以非診斷、非因果語言完成一頁摘要

### 🧭 闖關方式
1. 先按下方「🧰 魔法工具箱」那一格左邊的 ▶（第一次執行 Colab 會花幾秒鐘連線）。
2. 依序完成每個「🎯 任務」，再執行它下面的「檢查」格。
3. 看到 ✅ 就往下一個任務；看到 ❌ 就依提示修改並重跑。
4. 全部通過後，執行最下面的「🔑 通關密語」格。

> 💾 建議先在雲端硬碟中儲存副本。
> 🛡️ 本資料完全合成，輸出只用於課堂方法演練，不能取代醫療專業評估、診斷或處置。
> 🎲 切分請用 `random_state=42`；B-1 不填補，B-3 的 SimpleImputer 只從 train 學規則，B-5 才讀 final test 一次。

In [ ]:
#@title 🧰 魔法工具箱：先在右邊填「暱稱」，再按左邊的 ▶ 執行這一格 { display-mode: "form" }
暱稱 = "" #@param {type:"string"}
# ======================================================================
#  統計冒險之旅 2026 · 關卡檢查工具（看不懂沒關係，這一格不是今天的功課 😉）
# ======================================================================
import hashlib, unicodedata, io, sys, re, contextlib, traceback, builtins, math, warnings
warnings.filterwarnings("ignore")

_LEVEL = "B2"
_COURSE_NAMESPACE = "stats-quest-2026-datama"
_PREFIX = "SQ"
_TASKS = ["B-1", "B-2", "B-3", "B-4", "B-5"]
_XP_EACH = 40
_CHECKS = {}
_PASSED = builtins.__dict__.setdefault("_sq_" + _LEVEL, {})
_HINTS = {}

def _norm_name(s):
    return re.sub(r"\s+", "", unicodedata.normalize("NFKC", str(s))).lower()

def _squash(s):
    return re.sub(r"\s+", "", str(s))

def 出現(out, *subs):
    """輸出中是否（忽略空白）包含所有片段"""
    o = _squash(out)
    return all(_squash(x) in o for x in subs)

def 數字們(out):
    """抓出輸出裡所有的數字（float）"""
    return [float(x) for x in re.findall(r"-?\d+(?:\.\d+)?", str(out))]

# ---------------- 判分器 2.0 ----------------
class _Miss(Exception):
    pass

def 抓變數(ns, name, 型別=None):
    """從任務格執行後的變數取值；沒有就給友善訊息。"""
    if name not in ns:
        raise _Miss(f"我找不到變數 {name}，請確認你有把答案存進名字叫 {name} 的變數（大小寫要一樣）。")
    v = ns[name]
    if 型別 is not None and not isinstance(v, 型別):
        raise _Miss(f"{name} 的型別看起來不對（目前是 {type(v).__name__}）。")
    return v

def _num(v):
    try:
        import numpy as _np
        if hasattr(v, "item"): v = v.item()
    except Exception:
        pass
    return float(v)

def 約等於(v, 目標, 容差=None, 相對=0.01):
    """數值容差：|v-目標| <= 容差（預設為 目標 的 1%，且至少 1e-9）"""
    try:
        x = _num(v)
    except Exception:
        return False
    if x != x:   # NaN
        return False
    tol = 容差 if 容差 is not None else max(abs(目標) * 相對, 1e-9)
    return abs(x - 目標) <= tol

def 資料框像(obj, 列=None, 欄=None, 含欄位=None, 種類="DataFrame"):
    """檢查 DataFrame / Series：列數、欄數、必須包含的欄位；回傳 (ok, 訊息)"""
    import pandas as _pd
    if 種類 == "DataFrame" and not isinstance(obj, _pd.DataFrame):
        return False, f"這應該是一個 DataFrame（目前是 {type(obj).__name__}）。"
    if 種類 == "Series" and not isinstance(obj, _pd.Series):
        return False, f"這應該是一個 Series（目前是 {type(obj).__name__}）。"
    if 列 is not None and len(obj) != 列:
        return False, f"列數應該是 {列}，目前是 {len(obj)}。"
    if 欄 is not None and getattr(obj, "shape", (0, 0))[1] != 欄:
        return False, f"欄數應該是 {欄}，目前是 {obj.shape[1]}。"
    if 含欄位:
        cols = list(obj.columns) if hasattr(obj, "columns") else list(obj.index)
        missing = [c for c in 含欄位 if c not in cols]
        if missing:
            return False, "缺少欄位：" + "、".join(map(str, missing))
    return True, ""


class _NeedMoreInput(Exception):
    pass

_BUILTIN_NAMES = ("sum", "list", "dict", "set", "str", "int", "float", "max", "min", "len",
                  "print", "type", "range", "sorted", "abs", "round", "tuple", "map", "filter",
                  "open", "format", "all", "any", "zip", "bool", "next", "chr", "ord", "id")

_HIST = builtins.__dict__.setdefault("_sq_hist", [])
def _on_pre_run(*args):
    try:
        info = args[0]
        src = getattr(info, "raw_cell", None)
        if isinstance(src, str):
            _HIST.append(src)
    except Exception:
        pass
try:
    _ip = get_ipython()
    if not builtins.__dict__.get("_sq_hooked"):
        _ip.events.register("pre_run_cell", _on_pre_run)
        builtins.__dict__["_sq_hooked"] = True
except Exception:
    pass

def _history():
    try:
        ip = get_ipython()
        h = list(ip.user_ns.get("In") or ip.user_ns.get("_ih") or [])
    except Exception:
        h = list(globals().get("In") or [])
    return [c for c in (h + list(_HIST)) if isinstance(c, str)]

_CALL = re.compile(r"\s*(檢查|通關密語|全部檢查)\s*\(")

def _clean_cell(cell):
    return "\n".join(ln for ln in cell.splitlines() if not _CALL.match(ln))

def _is_mine(cell):
    s = cell.strip()
    if not s:
        return False
    if "#@title" in s or "任務定義(" in s or "_sq_" in s:
        return False
    if _CALL.match(s):
        return False
    return True

def _find_cells(tid):
    marker = "# 🎯 任務 " + tid
    marked = free = None
    im = ifree = -1
    for i, cell in enumerate(_history()):
        if not _is_mine(cell):
            continue
        if marker in cell:
            marked, im = cell, i
        elif "🎯 任務" not in cell:
            free, ifree = cell, i
    return marked, im, free, ifree

def _describe(src):
    body = [ln for ln in src.splitlines() if ln.strip() and not ln.strip().startswith("#")]
    if not body:
        return "（空白）"
    first = body[0].strip()
    return ("%s%s（共 %d 行）" % (first[:52], "…" if len(first) > 52 else "", len(body)))

def _fig_info(_plt):
    out = []
    try:
        for n in _plt.get_fignums():
            f = _plt.figure(n)
            for ax in f.get_axes():
                out.append(dict(title=ax.get_title() or "", xlabel=ax.get_xlabel() or "", ylabel=ax.get_ylabel() or "",
                                n_lines=len(ax.lines), n_patches=len(ax.patches), n_collections=len(ax.collections),
                                legend=bool(ax.get_legend())))
    except Exception:
        pass
    return out

def _make_runner(src):
    def run(*inputs):
        feed = iter([str(x) for x in inputs])
        buf = io.StringIO()
        try:
            ns = dict(get_ipython().user_ns)
        except Exception:
            ns = dict(globals())
        run.shadowed = []
        for _n in _BUILTIN_NAMES:
            _b = getattr(builtins, _n, None)
            if _n in ns and _b is not None and ns[_n] is not _b:
                ns.pop(_n, None)
                run.shadowed.append(_n)
        def _fake_input(prompt=""):
            try:
                return next(feed)
            except StopIteration:
                raise _NeedMoreInput()
        ns["input"] = _fake_input
        ns["__name__"] = "__main__"
        try:
            import matplotlib
            import matplotlib.pyplot as _plt
            _plt.close("all"); _orig_show = _plt.show; _plt.show = lambda *a, **k: None
        except Exception:
            _plt = None
        run.figs = []
        try:
            with contextlib.redirect_stdout(buf):
                exec(compile(src, "<任務 " + _LEVEL + ">", "exec"), ns)
        finally:
            if _plt is not None:
                run.figs = _fig_info(_plt)
                _plt.show = _orig_show
                _plt.close("all")
        return buf.getvalue(), ns
    run.src = src
    run.figs = []
    return run

def 任務定義(tid, fn, 提示=""):
    _CHECKS[tid] = fn
    _HINTS[tid] = 提示

def _fix_shadowed():
    try:
        ns = get_ipython().user_ns
    except Exception:
        ns = globals()
    bad = []
    for n in _BUILTIN_NAMES:
        b = builtins.__dict__.get(n)
        if b is not None and n in ns and ns[n] is not b:
            del ns[n]
            bad.append(n)
    return bad

def _progress():
    done = 0
    total = 0
    for t in _TASKS:
        total += 1
        if _PASSED.get(t):
            done += 1
    bar = "■" * done + "□" * (total - done)
    return f"[{bar}] {done}/{total}"

def _run_check(tid, src):
    run = _make_runner(src)
    try:
        result = _CHECKS[tid](run)
    except _NeedMoreInput:
        return False, "你的程式呼叫 input() 的次數比題目預期的多，請檢查輸入的次數。", []
    except _Miss as e:
        return False, str(e), getattr(run, "shadowed", [])
    except Exception:
        tb = traceback.format_exc().strip().splitlines()[-1]
        return False, "程式執行時發生錯誤 → " + tb, getattr(run, "shadowed", [])
    ok, extra = (result, "") if isinstance(result, bool) else result
    return ok, extra, getattr(run, "shadowed", [])

def _pass(tid):
    first = not _PASSED.get(tid)
    _PASSED[tid] = True
    print(f"✅ 任務 {tid} 通過！{'+' + str(_XP_EACH) + ' XP ' if first else ''}{_progress()}")

def 檢查(tid):
    _shadow = _fix_shadowed()
    tid = builtins.str(tid)
    if tid not in _CHECKS:
        print(f"⚠️ 找不到任務 {tid} 的檢查設定。"); return
    marked, im, free, ifree = _find_cells(tid)
    if marked is None and free is None:
        print(f"❌ 這次執行階段裡，我找不到你寫的程式。")
        print(f"   👉 請先按「# 🎯 任務 {tid}」那一格左邊的 ▶ 執行它，再執行這一格。")
        print("   （如果剛剛重新啟動過執行階段，上面每一格都要重跑一次，包含最上面的魔法工具箱）")
        return
    order = []
    if marked is not None:
        order.append(("標記", marked))
    if free is not None and ifree > im:
        order.append(("最後執行", free))
    if not order:
        order = [("最後執行", free)]
    tried = []
    for kind, src in order:
        ok, extra, shadowed = _run_check(tid, _clean_cell(src))
        tried.append((kind, src, extra, shadowed))
        if ok:
            _pass(tid)
            if extra:
                print("   💬 " + str(extra))
            if _shadow:
                print(f"   ℹ️ 你之前把內建名稱 {'、'.join(_shadow)} 拿來當變數名了，我已經幫你還原。")
                print("      建議換個名字（例如 total、items），不然後面的程式會出現很難懂的錯誤。")
            if kind == "最後執行":
                print(f"   ℹ️ 你的程式最上面少了「# 🎯 任務 {tid}」那一行，我是用你最後執行的那一格判分的。")
                print("      把那一行加回去，之後的檢查會更準確。")
            if shadowed:
                print(f"   ℹ️ 你之前把內建名稱 {'、'.join(shadowed)} 拿來當變數名了，判分時我先幫你還原。")
            if all(_PASSED.get(t) for t in _TASKS):
                print("🏆 本關所有任務都完成了！請執行最下面的「通關密語」那一格。")
            return
    kind, src, extra, shadowed = tried[0]
    print(f"❌ 任務 {tid} 還沒通過。{_progress()}")
    if extra:
        print("   💬 " + str(extra))
    if _HINTS.get(tid):
        print("   💡 提示：" + _HINTS[tid])
    _sh = _shadow + [n for n in shadowed if n not in _shadow]
    if _sh:
        print(f"   ⚠️ 你把內建名稱 {'、'.join(_sh)} 拿來當變數名了（我已還原），這會造成很難懂的錯誤，請改名後重跑那一格。")
    print("   🔎 我判分的是這一段程式：" + _describe(src))
    print(f"      如果這不是你剛剛寫的版本 → 確認第一行的「# 🎯 任務 {tid}」有保留，並重新執行那一格，再按檢查。")

def 全部檢查():
    """出錯或重新啟動執行階段後，重跑完所有任務格，再用這個一次驗收整關。"""
    _fix_shadowed()
    print(f"🔁 重新檢查 {_LEVEL} 的 {len(_TASKS)} 個任務…")
    todo = []
    for t in _TASKS:
        marked, im, free, ifree = _find_cells(t)
        if marked is None and free is None:
            todo.append(t)
            continue
        檢查(t)
    if todo:
        print("⏭️ 這次還沒執行過的任務：" + "、".join(todo))
        print("   先按那幾格左邊的 ▶ 執行，再回來執行 全部檢查()。")

def 通關密語():
    _fix_shadowed()
    missing = [t for t in _TASKS if not _PASSED.get(t)]
    if missing:
        print("🔒 還有任務未通過：" + "、".join(missing) + "　完成後再來拿密語吧！")
        return
    name = 暱稱.strip() if isinstance(暱稱, str) else ""
    if not name:
        name = input("請輸入你在入口網頁登錄的暱稱：").strip()
    if not name:
        print("⚠️ 暱稱不能是空白。"); return
    code = hashlib.sha256(f"{_COURSE_NAMESPACE}|{_LEVEL}|{_norm_name(name)}".encode("utf-8")).hexdigest()[:6].upper()
    print("=" * 46)
    print(f"🎉 恭喜 {name}！{_LEVEL} 通關！")
    print(f"🔑 通關密語：{_PREFIX}-{_LEVEL}-{code}")
    print("👉 回到入口網頁，把密語貼到這一關的「輸入通關密語」欄位。")
    print("=" * 46)

try:
    import numpy as _np_, pandas as _pd_
    _np_.random.seed(42)
except Exception:
    pass
print(f"🧰 魔法工具箱已準備好！本關有 {len(_TASKS)} 個任務：{'、'.join(_TASKS)}")
print("   做完每個任務後，執行它下方的「檢查」格；全部通過後執行最下方的「通關密語」。")

# ---------------- 各任務的檢查規則 ----------------
def _check_B_1(run):
    out, ns = run()
    if list(抓變數(ns, "缺值欄位")) != ["膽固醇"]: return (False, "只有 膽固醇 有缺值。")
    if int(抓變數(ns, "膽固醇缺值數")) != 8: return (False, "原始資料的 膽固醇 應保留 8 個缺值，不能先用全資料填補。")
    h = 抓變數(ns, "heart")
    if int(h["膽固醇"].isna().sum()) != 8: return (False, "B-1 只盤點缺值；填補要留給 B-3 的 Pipeline。")
    return (約等於(抓變數(ns, "資料陽性比例"), 0.57750, 0.001), "這是合成資料中的陽性比例，不是真實人口盛行率。")
任務定義("B-1", _check_B_1, 提示="先記錄缺值，不要 fillna；SimpleImputer 會在 B-3 的 Pipeline 只用 train 學中位數。")

def _check_B_2(run):
    out, ns = run()
    ok, msg = 資料框像(抓變數(ns, "X"), 列=400, 欄=9)
    if not ok: return (False, msg)
    if int(抓變數(ns, "特徵數")) != 9: return (False, "特徵數 = X.shape[1]。")
    if [int(抓變數(ns, n).sum()) for n in ["y_train", "y_dev", "y_test"]] != [139, 46, 46]:
        return (False, "兩次切分都要使用 stratify 與 random_state=42。")
    return (len(抓變數(ns, "X_train")) == 240 and len(抓變數(ns, "X_dev")) == 80 and len(抓變數(ns, "X_test")) == 80, "切分應為 train/dev/final test = 60%/20%/20%。")
任務定義("B-2", _check_B_2, 提示="先保留 20% final test，再把其餘資料切出 25% 作 dev；不要在這一步補值。")

def _check_B_3(run):
    if "X_test" in run.src or "y_test" in run.src:
        return (False, "建模與調整階段不能讀 final test；這題只 fit train、評估 dev。")
    out, ns = run()
    model = 抓變數(ns, "clf")
    if "columntransformer" not in getattr(model, "named_steps", {}): return (False, "補值、編碼與縮放必須放在 ColumnTransformer 裡。")
    pre = model.named_steps["columntransformer"]
    if set(pre.transformers_[0][2]) != set(抓變數(ns, "數值欄")): return (False, "數值欄必須由 X_train 判定。")
    if set(pre.transformers_[1][2]) != set(抓變數(ns, "類別欄")): return (False, "類別欄必須由 X_train 判定。")
    if "simpleimputer" not in pre.named_transformers_["num"].named_steps: return (False, "數值 SimpleImputer 必須在 Pipeline 內。")
    if "onehotencoder" not in pre.named_transformers_["cat"].named_steps: return (False, "OneHotEncoder 必須在 Pipeline 內。")
    if pre.named_transformers_["cat"].named_steps["onehotencoder"].handle_unknown != "ignore": return (False, "OneHotEncoder 要用 handle_unknown='ignore'，才能安全處理未見類別。")
    if not 約等於(抓變數(ns, "訓練集填補中位數"), 249.5, 1e-9): return (False, "填補器應只用 X_train 學到 膽固醇 中位數 249.5。")
    if not 約等於(抓變數(ns, "devAUC"), 0.90601, 0.005): return (False, "devAUC = roc_auc_score(y_dev, dev機率)。")
    return (約等於(抓變數(ns, "dev準確率"), 0.82500, 0.01), "dev準確率使用 0.5 門檻；不可用 final test 調整。")
任務定義("B-3", _check_B_3, 提示="ColumnTransformer 在 Pipeline 內完成 impute／encode／scale；整體只 fit(X_train, y_train)。")

def _check_B_4(run):
    if "X_test" in run.src or "y_test" in run.src:
        return (False, "建議門檻只能從 dev 選，final test 要留到 B-5。")
    out, ns = run()
    r = 抓變數(ns, "dev召回率們", dict); p = 抓變數(ns, "dev精確率們", dict)
    if not 約等於(r[0.3], 0.93478, 0.005): return (False, "dev召回率們[0.3] 不對。")
    if not 約等於(p[0.3], 0.79630, 0.005): return (False, "dev精確率們要用 precision_score(y_dev, p)。")
    return (約等於(抓變數(ns, "建議門檻"), 0.3, 1e-6), "建議門檻 = dev 召回率 ≥ 0.9 的最高門檻。")
任務定義("B-4", _check_B_4, 提示="門檻比較全部使用 dev機率 與 y_dev。")

def _check_B_5(run):
    out, ns = run()
    top = list(抓變數(ns, "前三因子"))
    if set(top) != set(['胸痛類型_無症狀', 'ST下降', '運動心絞痛']): return (False, "前三因子是模型係數絕對值最大的三個預測欄位，不能說成致病原因。")
    if not 約等於(抓變數(ns, "最終測試AUC"), 0.88363, 0.005): return (False, "選好模型與門檻後，才在 final test 報告一次 AUC。")
    if not 約等於(抓變數(ns, "最終測試召回率"), 0.91304, 0.005): return (False, "final test 請套用 dev 選好的門檻，不可再調整。")
    if not 約等於(抓變數(ns, "最終測試精確率"), 0.77778, 0.005): return (False, "最終測試精確率不對。")
    d = 抓變數(ns, "摘要", dict)
    for k in ["開發集AUC", "建議門檻", "最終測試AUC", "最終測試召回率", "最終測試精確率", "前三因子", "用途邊界", "建議"]:
        if k not in d: return (False, f"摘要 缺少「{k}」。")
    boundary = str(d["用途邊界"])
    if "教學" not in boundary or "不能取代" not in boundary: return (False, "用途邊界需明說是合成資料教學演練，不能取代專業評估。")
    return (isinstance(d["建議"], str) and len(d["建議"]) >= 10, "建議要是一句至少 10 個字、且不做診斷或因果宣稱的話。")
任務定義("B-5", _check_B_5, 提示="final test 只在本題評一次；係數稱為預測訊號，不稱為致病原因。")

## ⚔️ Boss 戰：心臟病風險預警
「勇者健檢中心」用 400 筆固定種子合成資料示範風險排序。情境假設漏掉高風險個案的代價較高，因此以召回率優先比較門檻。

資料：`https://raw.githubusercontent.com/johnnychao/stats-quest-2026/v1.0.0/data/heart_check.csv`

> Day 2 的總驗收：缺值盤點 → 原始欄位三向切分 → ColumnTransformer 在 train 內補值／編碼／縮放 → train 建模 → dev 選門檻 → final test 一次報告 → 安全摘要。
>
> 這不是臨床模型；「模型較依賴的預測欄位」不等於致病原因。

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, recall_score, precision_score, confusion_matrix, roc_curve
heart = pd.read_csv("https://raw.githubusercontent.com/johnnychao/stats-quest-2026/v1.0.0/data/heart_check.csv")
print(heart.shape); print(heart.isna().sum())
heart.head()

In [ ]:
#@title 🈶 中文字型設定（畫圖前先執行；Colab 初次約 20～40 秒）
import glob, shutil, subprocess, sys, matplotlib
from matplotlib import font_manager

_font_globs = [
    '/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc',
    '/usr/share/fonts/opentype/noto/NotoSansCJK*.otf',
    'C:/Windows/Fonts/msjh*.ttc',
]
if sys.platform.startswith('linux') and shutil.which('apt-get'):
    if not any(glob.glob(pattern) for pattern in _font_globs[:2]):
        try:
            subprocess.run(
                ['apt-get', '-qq', 'install', '-y', 'fonts-noto-cjk'],
                check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
            )
        except (FileNotFoundError, subprocess.CalledProcessError) as error:
            raise RuntimeError('無法自動安裝中文字型；請確認網路後重新執行本格。') from error
for pattern in _font_globs:
    for path in glob.glob(pattern):
        try:
            font_manager.fontManager.addfont(path)
        except (OSError, RuntimeError):
            pass

_available_fonts = {font.name for font in font_manager.fontManager.ttflist}
_preferred_fonts = [
    'Noto Sans TC', 'Noto Sans CJK TC',
    'Microsoft JhengHei', 'Microsoft JhengHei UI', 'PingFang TC',
    'Noto Sans CJK JP', 'Arial Unicode MS',
]
_chinese_font = next((name for name in _preferred_fonts if name in _available_fonts), None)
if _chinese_font is None:
    raise RuntimeError('找不到可顯示繁體中文的字型；請安裝 Noto Sans CJK 後重新執行本格。')
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.sans-serif'] = [_chinese_font, 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False
print(f"✅ 中文字型設定完成：{_chinese_font}")

### 🎯 任務 B-1　缺值盤點與資料陽性比例

找出有缺值的欄位名稱 `缺值欄位`、計算 `膽固醇缺值數` 與 `資料陽性比例`。此時不要 `fillna`；填補規則必須留在後續 Pipeline，避免從 dev 或 final test 偷學中位數。

In [ ]:
# 🎯 任務 B-1　缺值與盛行率（請保留這一行）
heart = pd.read_csv("https://raw.githubusercontent.com/johnnychao/stats-quest-2026/v1.0.0/data/heart_check.csv")
缺值欄位 = heart.columns[heart.isna().sum() > 0].tolist()
膽固醇缺值數 = int(heart["膽固醇"].isna().sum())
資料陽性比例 = ???
print(缺值欄位, 膽固醇缺值數, round(資料陽性比例, 3))

In [ ]:
檢查("B-1")   # ◀ 執行這一格，看看任務 B-1 有沒有過關

### 🎯 任務 B-2　先切原始欄位，再交給 Pipeline

建立原始 `X`（只去掉編號、心臟病，不先補值、不先 one-hot）與 `y`。先保留 20% final test，再把其餘資料切出 25% 作 dev；兩次都用 `random_state=42` 與 `stratify`。把原始欄數存成 `特徵數`。

In [ ]:
# 🎯 任務 B-2　編碼與切分（請保留這一行）
X = heart.drop(columns=["編號", "心臟病"])
y = heart["心臟病"]
X_work, X_test, y_work, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=???)
X_train, X_dev, y_train, y_dev = train_test_split(X_work, y_work, test_size=0.25, random_state=42, stratify=???)
特徵數 = ???
print(X.dtypes); print(特徵數, X_train.shape, X_dev.shape, X_test.shape)

In [ ]:
檢查("B-2")   # ◀ 執行這一格，看看任務 B-2 有沒有過關

### 🎯 任務 B-3　ColumnTransformer 內完成補值、編碼與縮放

先由 `X_train` 找出 `數值欄`、`類別欄`。建立 `前處理`：數值欄走 SimpleImputer(strategy='median') → StandardScaler；類別欄走 SimpleImputer(strategy='most_frequent') → OneHotEncoder(drop='first', handle_unknown='ignore') → StandardScaler(with_mean=False)。再以 `make_pipeline(前處理, LogisticRegression(max_iter=1000))` 建立 `clf`，全流程只 `.fit(X_train, y_train)`。

用 dev 算 `dev機率`、`devAUC`、`dev準確率`；並由 fitted 數值流程讀取膽固醇的 `訓練集填補中位數`。這一題不可讀 final test。

In [ ]:
# 🎯 任務 B-3　建模與評估（請保留這一行）
數值欄 = X_train.select_dtypes(exclude="object").columns.tolist()
類別欄 = X_train.select_dtypes(include="object").columns.tolist()
數值流程 = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
類別流程 = make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(drop="first", handle_unknown="ignore"), StandardScaler(with_mean=False))
前處理 = ColumnTransformer([("num", 數值流程, 數值欄), ("cat", 類別流程, 類別欄)], verbose_feature_names_out=False)
clf = make_pipeline(前處理, LogisticRegression(max_iter=1000)).fit(X_train, y_train)
dev機率 = clf.predict_proba(X_dev)[:, 1]
devAUC = ???
dev準確率 = ((dev機率 >= 0.5).astype(int) == y_dev).mean()
膽固醇位置 = 數值欄.index("膽固醇")
訓練集填補中位數 = clf.named_steps["columntransformer"].named_transformers_["num"].named_steps["simpleimputer"].statistics_[膽固醇位置]
print(round(devAUC, 3), round(dev準確率, 3), 訓練集填補中位數)

In [ ]:
檢查("B-3")   # ◀ 執行這一格，看看任務 B-3 有沒有過關

### 🎯 任務 B-4　在 dev 選門檻

只用 `dev機率` 與 `y_dev`，對 `[0.5, 0.4, 0.3, 0.2]` 算 `dev召回率們`、`dev精確率們`；`建議門檻` 是 dev 召回率 ≥ 0.9 的最高門檻。final test 仍保持封存。

In [ ]:
# 🎯 任務 B-4　門檻取捨：漏掉病人的代價（請保留這一行）
門檻們 = [0.5, 0.4, 0.3, 0.2]
dev召回率們, dev精確率們 = {}, {}
for t in 門檻們:
    p = (dev機率 >= t).astype(int)
    dev召回率們[t] = recall_score(y_dev, p)
    dev精確率們[t] = ???
建議門檻 = max(t for t in 門檻們 if dev召回率們[t] >= 0.9)
for t in 門檻們:
    print(f"dev 門檻 {t}：召回率 {dev召回率們[t]:.3f}　精確率 {dev精確率們[t]:.3f}")
print("建議門檻", 建議門檻)

In [ ]:
檢查("B-4")   # ◀ 執行這一格，看看任務 B-4 有沒有過關

### 🎯 任務 B-5　final test 一次報告與安全摘要

模型與門檻都鎖定後，第一次也是唯一一次用 `X_test/y_test` 報告 `最終測試AUC`、`最終測試召回率`、`最終測試精確率`。再從 fitted `ColumnTransformer.get_feature_names_out()` 取得轉換後欄名，與 Logistic 係數對齊建立 `前三因子`；它們只稱為預測訊號，不作因果或診斷解讀。`摘要` 必須含用途邊界。

In [ ]:
# 🎯 任務 B-5　給醫師的一頁摘要（請保留這一行）
最終機率 = clf.predict_proba(X_test)[:, 1]
最終測試AUC = roc_auc_score(y_test, 最終機率)
最終預測 = (最終機率 >= 建議門檻).astype(int)
最終測試召回率 = recall_score(y_test, 最終預測)
最終測試精確率 = precision_score(y_test, 最終預測)
轉換後欄名 = clf.named_steps["columntransformer"].get_feature_names_out()
係數 = pd.Series(clf.named_steps["logisticregression"].coef_[0], index=轉換後欄名)
前三因子 = 係數.abs().sort_values(ascending=False).head(3).index.tolist()
摘要 = {
    "開發集AUC": round(devAUC, 3),
    "建議門檻": 建議門檻,
    "最終測試AUC": round(最終測試AUC, 3),
    "最終測試召回率": round(最終測試召回率, 3),
    "最終測試精確率": round(最終測試精確率, 3),
    "前三因子": 前三因子,
    "用途邊界": ???,
    "建議": ???,
}
for k, v in 摘要.items():
    print(f"{k}：{v}")

In [ ]:
檢查("B-5")   # ◀ 執行這一格，看看任務 B-5 有沒有過關

## 🎤 成果分享（3 分鐘）
向「健檢中心主管」報告：模型只在合成資料上示範什麼？前處理如何只從 train 學規則？門檻如何在 dev 選定？final test 的一次結果如何？哪些限制使它不能成為醫療工具？

## 🌟 進階挑戰（不計分）
1. 只在 train 內做五摺交叉驗證；完整 ColumnTransformer 與 LogisticRegression 必須留在同一 Pipeline。
2. 在 dev 畫 ROC 並標出建議門檻；不要再查看 final test 或依其結果改門檻。

---
## 🔑 通關密語
　Day 2 通關！你已經會建模、評估，也會替決策者選門檻了。
全部任務都 ✅ 之後，執行下面這一格，會得到你專屬的通關密語（和暱稱綁定，每個人不一樣）。

In [ ]:
通關密語()

---
### 🧭 接下來
**下一關：🔁 L07 交叉驗證與隨機森林** → [在 Colab 開啟](https://colab.research.google.com/github/johnnychao/stats-quest-2026/blob/v1.0.0/notebooks/L07_cv_forest.ipynb)

回到入口網頁：https://johnnychao.github.io/stats-quest-2026/